# 05 - So sánh mô hình và kiểm định thống kê

Yêu cầu: so sánh mô hình bằng Cross Validation, F-test, t-test và ANOVA nếu phù hợp.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for base in candidates:
        if (base / "data" / "wage_model.csv").exists():
            return base
        nested = base / "Machine Learning"
        if (nested / "data" / "wage_model.csv").exists():
            return nested
    raise FileNotFoundError("Không tìm thấy data/wage_model.csv. Hãy mở notebook trong folder Machine Learning.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "model_results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

eda_df = pd.read_csv(DATA_DIR / "wage_eda.csv")
model_df = pd.read_csv(DATA_DIR / "wage_model.csv")

print("Project root found.")
print("EDA data:", eda_df.shape)
print("Model data:", model_df.shape)


Project root found.
EDA data: (3010, 30)
Model data: (3010, 38)


In [2]:
target = "lwage"

simple_features = ["educ"]
extended_features = [
    "educ", "exper", "expersq", "IQ", "KWW",
    "black", "south", "smsa", "fatheduc", "motheduc",
    "married_2", "married_3", "married_4", "married_5", "married_6",
]
full_features = [col for col in model_df.columns if col != target]

feature_groups = {
    "M1_simple": simple_features,
    "M2_extended": extended_features,
    "M3_full": full_features,
}

X_full = model_df[full_features]
y = model_df[target]

print("Target:", target)
print("M1 features:", len(simple_features))
print("M2 features:", len(extended_features))
print("M3 features:", len(full_features))


Target: lwage
M1 features: 1
M2 features: 15
M3 features: 37


In [3]:
region_cols = sorted([col for col in model_df.columns if col.startswith("reg66")])
region66 = pd.Series(1, index=model_df.index, name="region66")

for col in region_cols:
    region_code = int(col.replace("reg66", ""))
    region66.loc[model_df[col] == 1] = region_code

display(region66.value_counts().sort_index().rename("n_obs").to_frame())


,n_obs
region66,
1,140
2,484
3,589
4,193
5,627
6,289
7,331
8,85
9,272


## 5.1 F-test cho các mô hình OLS lồng nhau


In [4]:
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm
from scipy import stats


def fit_ols(features):
    X = sm.add_constant(model_df[features], has_constant="add")
    return sm.OLS(model_df[target], X).fit()


ols_simple = fit_ols(simple_features)
ols_extended = fit_ols(extended_features)
ols_full = fit_ols(full_features)

f_tests = []
for small_name, big_name, small_model, big_model in [
    ("M1_simple", "M2_extended", ols_simple, ols_extended),
    ("M2_extended", "M3_full", ols_extended, ols_full),
    ("M1_simple", "M3_full", ols_simple, ols_full),
]:
    f_stat, p_value, df_diff = big_model.compare_f_test(small_model)
    f_tests.append({
        "restricted_model": small_name,
        "unrestricted_model": big_name,
        "f_stat": f_stat,
        "p_value": p_value,
        "df_diff": df_diff,
        "decision_5pct": "reject H0" if p_value < 0.05 else "fail to reject H0",
    })

f_test_table = pd.DataFrame(f_tests)
display(f_test_table.round(4))
display(anova_lm(ols_simple, ols_extended, ols_full))
f_test_table.to_csv(RESULT_DIR / "ols_nested_f_tests.csv", index=False)


,restricted_model,unrestricted_model,f_stat,p_value,df_diff,decision_5pct
0,M1_simple,M2_extended,71.9205,0.0,14.0,reject H0
1,M2_extended,M3_full,2.7118,0.0,21.0,reject H0
2,M1_simple,M3_full,30.7407,0.0,35.0,reject H0


,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,3008.0,534.126258,0.0,NaN,NaN,NaN
1,2994.0,399.704805,14.0,134.421453,72.784087,1.558469e-178
2,2973.0,392.192264,21.0,7.512540,2.711836,4.131887e-05


## 5.2 t-test cho hệ số trong OLS đầy đủ


In [5]:
key_terms = [
    "educ", "exper", "expersq", "IQ", "KWW", "black", "south", "smsa",
    "fatheduc", "motheduc", "married_2", "married_3", "married_4",
    "married_5", "married_6",
]

t_test_coef = pd.DataFrame({
    "term": key_terms,
    "coef": ols_full.params.reindex(key_terms),
    "t_stat": ols_full.tvalues.reindex(key_terms),
    "p_value": ols_full.pvalues.reindex(key_terms),
})
t_test_coef["decision_5pct"] = np.where(t_test_coef["p_value"] < 0.05, "reject H0", "fail to reject H0")

display(t_test_coef.sort_values("p_value").round(4))
t_test_coef.to_csv(RESULT_DIR / "ols_full_coefficient_t_tests.csv", index=False)


,term,coef,t_stat,p_value,decision_5pct
educ,educ,0.0528,12.0697,0.0000,reject H0
married_6,married_6,-0.1794,-9.9994,0.0000,reject H0
exper,exper,0.0630,9.0250,0.0000,reject H0
smsa,smsa,0.1394,7.0699,0.0000,reject H0
black,black,-0.1253,-6.1080,0.0000,reject H0
KWW,KWW,0.0066,5.9164,0.0000,reject H0
south,south,-0.1459,-5.7274,0.0000,reject H0
expersq,expersq,-0.0018,-5.6433,0.0000,reject H0
married_5,married_5,-0.0889,-2.3785,0.0174,reject H0
married_4,married_4,-0.0605,-1.9871,0.0470,reject H0


## 5.3 ANOVA theo nhóm học vấn


In [6]:
anova_df = model_df.copy()
anova_df["educ_group"] = pd.cut(
    anova_df["educ"],
    bins=[0, 11, 12, 15, np.inf],
    labels=["<=11", "12", "13-15", ">=16"],
    include_lowest=True,
)

anova_groups = [
    group[target].values
    for _, group in anova_df.groupby("educ_group", observed=True)
]
anova_stat, anova_p = stats.f_oneway(*anova_groups)

anova_summary = (
    anova_df
    .groupby("educ_group", observed=True)[target]
    .agg(["count", "mean", "std"])
    .reset_index()
)

display(anova_summary.round(4))
print(f"ANOVA F-stat = {anova_stat:.4f}, p-value = {anova_p:.4g}")
print("Decision:", "reject H0" if anova_p < 0.05 else "fail to reject H0")
anova_summary.to_csv(RESULT_DIR / "lwage_anova_by_educ_group.csv", index=False)


,educ_group,count,mean,std
0,<=11,497,5.9933,0.4248
1,12,992,6.2490,0.4191
2,13-15,704,6.2765,0.4077
3,>=16,817,6.4281,0.4336


ANOVA F-stat = 110.2685, p-value = 9.808e-68
Decision: reject H0


## 5.4 GroupKFold Cross Validation cho Linear/Ridge/Lasso


In [7]:
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

alphas = np.logspace(-4, 4, 100)
ml_models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", RidgeCV(alphas=alphas, cv=5))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", LassoCV(alphas=alphas, cv=5, max_iter=100000, random_state=42))]),
}

cv = GroupKFold(n_splits=5)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

cv_rows = []
cv_rmse_by_model = {}

for name, estimator in ml_models.items():
    scores = cross_validate(estimator, X_full, y, cv=cv, groups=region66, scoring=scoring)
    rmse = -scores["test_rmse"]
    mae = -scores["test_mae"]
    r2 = scores["test_r2"]
    cv_rmse_by_model[name] = rmse
    cv_rows.append({
        "model": name,
        "rmse_mean": rmse.mean(),
        "rmse_std": rmse.std(ddof=1),
        "mae_mean": mae.mean(),
        "mae_std": mae.std(ddof=1),
        "r2_mean": r2.mean(),
        "r2_std": r2.std(ddof=1),
    })

cv_table = pd.DataFrame(cv_rows).sort_values("rmse_mean")
display(cv_table.round(4))
cv_table.to_csv(RESULT_DIR / "ml_groupkfold_cv_metrics.csv", index=False)


,model,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std
2,Lasso,0.3713,0.0122,0.2906,0.0100,0.2541,0.0922
1,Ridge,0.3747,0.0125,0.2946,0.0112,0.2405,0.0964
0,Linear Regression,0.3774,0.0134,0.2976,0.0129,0.2288,0.1063


## 5.5 Paired t-test giữa RMSE các mô hình ML


In [8]:
paired_rows = []
base_model = "Linear Regression"

for other_model in ["Ridge", "Lasso"]:
    t_stat, p_value = stats.ttest_rel(cv_rmse_by_model[base_model], cv_rmse_by_model[other_model])
    paired_rows.append({
        "model_a": base_model,
        "model_b": other_model,
        "mean_rmse_a": cv_rmse_by_model[base_model].mean(),
        "mean_rmse_b": cv_rmse_by_model[other_model].mean(),
        "mean_diff_a_minus_b": (cv_rmse_by_model[base_model] - cv_rmse_by_model[other_model]).mean(),
        "t_stat": t_stat,
        "p_value": p_value,
        "decision_5pct": "reject H0" if p_value < 0.05 else "fail to reject H0",
    })

paired_t_table = pd.DataFrame(paired_rows)
display(paired_t_table.round(4))
paired_t_table.to_csv(RESULT_DIR / "ml_cv_paired_t_tests.csv", index=False)


,model_a,model_b,mean_rmse_a,mean_rmse_b,mean_diff_a_minus_b,t_stat,p_value,decision_5pct
0,Linear Regression,Ridge,0.3774,0.3747,0.0027,2.3853,0.0756,fail to reject H0
1,Linear Regression,Lasso,0.3774,0.3713,0.0060,2.4369,0.0714,fail to reject H0


## Kết luận gợi ý

- Dùng F-test để xem mô hình nhiều biến có cải thiện đáng kể so với mô hình ít biến không.
- Dùng t-test trong OLS để xem hệ số từng biến có ý nghĩa thống kê không.
- Dùng ANOVA để kiểm tra `lwage` trung bình có khác nhau giữa các nhóm học vấn không.
- Dùng GroupKFold để so sánh Linear Regression, Ridge và Lasso theo cách không trộn vùng giữa train/test.
